<a href="https://colab.research.google.com/github/Ashu-42/quant_research_crypto_volatility_forecasting/blob/quant_DL_ashu/notebooks/15min/01_Data_Creation_Intraday.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Intraday Crypto Data Creation

New intraday branch for the volatility-forecasting project.

**Default:** 15-minute Binance Spot candles for BTCUSDT, ETHUSDT, SOLUSDT and XRPUSDT.

All new data/artifacts are kept under:

`/content/drive/MyDrive/Quant Research/15min`

The code is interval-aware. To rerun the same raw-data pipeline at 1-hour frequency later, change only:

```python
KLINE_INTERVAL = "1h"
```

Interval-specific subfolders prevent 15m and 1h files from overwriting each other.

This notebook only creates and audits raw OHLCV data. Returns, next-candle RV targets, intraday seasonality and model-ready features belong in Notebook 02.

In [1]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, time, warnings

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configuration

In [2]:
# =========================
# USER CONFIGURATION
# =========================
PROJECT_ROOT = Path("/content/drive/MyDrive/Quant Research/15min")

SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT"]

# Change to "1h" later for an hourly-candle experiment.
KLINE_INTERVAL = "15m"

# UTC calendar range. END_DATE is inclusive.
# Kept aligned with the original daily project.
START_DATE = "2017-08-17"
END_DATE = "2026-08-01"

REQUEST_LIMIT = 1000
REQUEST_TIMEOUT_SECONDS = 30
REQUEST_SLEEP_SECONDS = 0.08

CHECKPOINT_EVERY_BATCHES = 50
RESUME_IF_CHECKPOINT_EXISTS = True

BASE_URL = "https://data-api.binance.vision"
KLINES_ENDPOINT = f"{BASE_URL}/api/v3/klines"
TIME_ENDPOINT = f"{BASE_URL}/api/v3/time"
EXCHANGE_INFO_ENDPOINT = f"{BASE_URL}/api/v3/exchangeInfo"

# Fixed-duration Binance Spot intervals.
INTERVAL_TO_MINUTES = {
    "1m":1, "3m":3, "5m":5, "15m":15, "30m":30,
    "1h":60, "2h":120, "4h":240, "6h":360,
    "8h":480, "12h":720, "1d":1440, "3d":4320,
    "1w":10080,
}

if KLINE_INTERVAL not in INTERVAL_TO_MINUTES:
    raise ValueError(
        f"Unsupported fixed interval {KLINE_INTERVAL}. "
        f"Allowed: {sorted(INTERVAL_TO_MINUTES)}"
    )

INTERVAL_MINUTES = INTERVAL_TO_MINUTES[KLINE_INTERVAL]
INTERVAL_MS = INTERVAL_MINUTES * 60 * 1000

RAW_ROOT = PROJECT_ROOT / "data" / "raw" / KLINE_INTERVAL
MANIFEST_ROOT = PROJECT_ROOT / "manifests" / KLINE_INTERVAL
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / KLINE_INTERVAL
MODEL_ROOT = PROJECT_ROOT / "models" / KLINE_INTERVAL
RESULTS_ROOT = PROJECT_ROOT / "results" / KLINE_INTERVAL

for p in [RAW_ROOT, MANIFEST_ROOT, ARTIFACT_ROOT, MODEL_ROOT, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

symbol_tag = "_".join(s.replace("USDT","").lower() for s in SYMBOLS)

COMBINED_RAW_PATH = RAW_ROOT / f"crypto_binance_{KLINE_INTERVAL}_{symbol_tag}_raw.parquet"
GAP_AUDIT_PATH = MANIFEST_ROOT / f"gap_audit_{KLINE_INTERVAL}.csv"

CANDLE_STRUCTURE_AUDIT_PATH = (
    MANIFEST_ROOT
    / f"candle_structure_audit_{KLINE_INTERVAL}.csv"
)

MANIFEST_PATH = MANIFEST_ROOT / f"crypto_binance_{KLINE_INTERVAL}_{symbol_tag}_raw_manifest.json"

print("Project root:", PROJECT_ROOT)
print("Interval:", KLINE_INTERVAL, f"({INTERVAL_MINUTES} minutes)")
print("Combined output:", COMBINED_RAW_PATH)

Project root: /content/drive/MyDrive/Quant Research/15min
Interval: 15m (15 minutes)
Combined output: /content/drive/MyDrive/Quant Research/15min/data/raw/15m/crypto_binance_15m_btc_eth_sol_xrp_raw.parquet


## Robust public-market-data session and effective candle range

In [3]:
retry_strategy = Retry(
    total=8, connect=8, read=8, status=8,
    backoff_factor=1.0,
    status_forcelist=[429,500,502,503,504],
    allowed_methods=["GET"],
    respect_retry_after_header=True,
    raise_on_status=False,
)

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry_strategy))
session.headers.update({"User-Agent":"QuantResearchIntradayVolatility/1.0"})

def ts_ms(ts):
    return int(pd.Timestamp(ts).timestamp() * 1000)

def ceil_ms(x, step):
    return ((x + step - 1) // step) * step

def floor_ms(x, step):
    return (x // step) * step

resp = session.get(TIME_ENDPOINT, timeout=REQUEST_TIMEOUT_SECONDS)
resp.raise_for_status()
server_ms = int(resp.json()["serverTime"])

requested_start = pd.Timestamp(START_DATE, tz="UTC")
requested_end_exclusive = pd.Timestamp(END_DATE, tz="UTC") + pd.Timedelta(days=1)

first_requested_open_ms = ceil_ms(ts_ms(requested_start), INTERVAL_MS)
last_requested_open_ms = floor_ms(ts_ms(requested_end_exclusive) - 1, INTERVAL_MS)

# Last candle that has definitely finished at Binance server time.
last_completed_open_ms = (server_ms // INTERVAL_MS) * INTERVAL_MS - INTERVAL_MS
effective_last_open_ms = min(last_requested_open_ms, last_completed_open_ms)

if effective_last_open_ms < first_requested_open_ms:
    raise ValueError("No completed candles fall inside the requested range.")

print("Binance server time:",
      pd.to_datetime(server_ms, unit="ms", utc=True))
print("First requested open:",
      pd.to_datetime(first_requested_open_ms, unit="ms", utc=True))
print("Last completed open used:",
      pd.to_datetime(effective_last_open_ms, unit="ms", utc=True))

Binance server time: 2026-08-21 18:53:46.406000+00:00
First requested open: 2017-08-17 00:00:00+00:00
Last completed open used: 2026-08-01 23:45:00+00:00


## Endpoint / symbol smoke test

In [4]:
symbol_checks = []

for symbol in SYMBOLS:
    r = session.get(
        EXCHANGE_INFO_ENDPOINT,
        params={"symbol":symbol},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    r.raise_for_status()
    payload = r.json()
    entries = payload.get("symbols", [])
    if len(entries) != 1:
        raise ValueError(f"Unexpected exchangeInfo response for {symbol}: {payload}")
    info = entries[0]
    symbol_checks.append({
        "symbol":symbol,
        "status":info.get("status"),
        "base_asset":info.get("baseAsset"),
        "quote_asset":info.get("quoteAsset"),
    })

symbol_validation_df = pd.DataFrame(symbol_checks)
display(symbol_validation_df)

if not symbol_validation_df["quote_asset"].eq("USDT").all():
    raise ValueError("Configured symbols are expected to be USDT pairs.")

smoke = session.get(
    KLINES_ENDPOINT,
    params={"symbol":SYMBOLS[0], "interval":KLINE_INTERVAL, "limit":2},
    timeout=REQUEST_TIMEOUT_SECONDS,
)
smoke.raise_for_status()
smoke_payload = smoke.json()

if (
    not isinstance(smoke_payload, list)
    or not smoke_payload
    or len(smoke_payload[0]) != 12
):
    raise ValueError("Unexpected Binance kline response structure.")

print("Endpoint and symbol validation passed.")

,symbol,status,base_asset,quote_asset
0,BTCUSDT,TRADING,BTC,USDT
1,ETHUSDT,TRADING,ETH,USDT
2,SOLUSDT,TRADING,SOL,USDT
3,XRPUSDT,TRADING,XRP,USDT


Endpoint and symbol validation passed.


## Kline parser and resume-safe pagination

In [5]:
KLINE_COLUMNS = [
    "open_time_ms","open","high","low","close","volume",
    "close_time_ms","quote_asset_volume","number_of_trades",
    "taker_buy_base_asset_volume","taker_buy_quote_asset_volume","unused",
]

FLOAT_COLUMNS = [
    "open","high","low","close","volume","quote_asset_volume",
    "taker_buy_base_asset_volume","taker_buy_quote_asset_volume",
]

FINAL_COLUMNS = [
    "symbol","interval","open_time","close_time",
    "open_time_ms","close_time_ms","open","high","low","close",
    "volume","quote_asset_volume","number_of_trades",
    "taker_buy_base_asset_volume","taker_buy_quote_asset_volume",
]

def parse_batch(rows, symbol):
    f = pd.DataFrame(rows, columns=KLINE_COLUMNS).drop(columns="unused")
    f["open_time_ms"] = pd.to_numeric(f["open_time_ms"], errors="raise").astype("int64")
    f["close_time_ms"] = pd.to_numeric(f["close_time_ms"], errors="raise").astype("int64")
    f["number_of_trades"] = pd.to_numeric(f["number_of_trades"], errors="raise").astype("int64")
    for c in FLOAT_COLUMNS:
        f[c] = pd.to_numeric(f[c], errors="raise").astype("float64")
    f["open_time"] = pd.to_datetime(f["open_time_ms"], unit="ms", utc=True)
    f["close_time"] = pd.to_datetime(f["close_time_ms"], unit="ms", utc=True)
    f["symbol"] = symbol
    f["interval"] = KLINE_INTERVAL
    return f[FINAL_COLUMNS]

def checkpoint_path(symbol):
    return RAW_ROOT / f"{symbol}_{KLINE_INTERVAL}_raw.parquet"

def sanitize_checkpoint(f, symbol):
    missing = set(FINAL_COLUMNS) - set(f.columns)
    if missing:
        raise ValueError(f"{symbol} checkpoint missing columns: {missing}")
    f = f.loc[
        f["symbol"].eq(symbol)
        & f["interval"].eq(KLINE_INTERVAL)
        & f["open_time_ms"].between(first_requested_open_ms, effective_last_open_ms)
    ].copy()
    f["open_time"] = pd.to_datetime(f["open_time"], utc=True, errors="raise")
    f["close_time"] = pd.to_datetime(f["close_time"], utc=True, errors="raise")
    return (
        f.sort_values("open_time_ms")
         .drop_duplicates("open_time_ms", keep="last")
         .reset_index(drop=True)
    )

def save_checkpoint(f, symbol):
    clean = (
        f.sort_values("open_time_ms")
         .drop_duplicates("open_time_ms", keep="last")
         .reset_index(drop=True)
    )
    clean.to_parquet(checkpoint_path(symbol), index=False)

def pull_symbol(symbol):
    existing = pd.DataFrame(columns=FINAL_COLUMNS)
    cursor = first_requested_open_ms
    cp = checkpoint_path(symbol)

    if RESUME_IF_CHECKPOINT_EXISTS and cp.exists():
        loaded = sanitize_checkpoint(pd.read_parquet(cp), symbol)
        if not loaded.empty and int(loaded["open_time_ms"].min()) <= first_requested_open_ms:
            existing = loaded
            cursor = int(existing["open_time_ms"].max()) + INTERVAL_MS
            print(f"{symbol}: resuming with {len(existing):,} rows; next open "
                  f"{pd.to_datetime(cursor, unit='ms', utc=True)}")
        elif not loaded.empty:
            print(f"{symbol}: existing checkpoint begins too late; restarting.")

    if cursor > effective_last_open_ms:
        print(f"{symbol}: checkpoint already covers requested range.")
        return existing

    batches = []
    batch_no = 0
    new_rows = 0

    while cursor <= effective_last_open_ms:
        params = {
            "symbol":symbol,
            "interval":KLINE_INTERVAL,
            "startTime":int(cursor),
            # Binance endTime is inclusive.
            "endTime":int(effective_last_open_ms + INTERVAL_MS - 1),
            "limit":REQUEST_LIMIT,
        }

        r = session.get(KLINES_ENDPOINT, params=params, timeout=REQUEST_TIMEOUT_SECONDS)

        if r.status_code == 418:
            raise RuntimeError(
                "Binance returned HTTP 418 (temporary IP ban). "
                "Stop and retry later."
            )

        r.raise_for_status()
        payload = r.json()

        if not isinstance(payload, list):
            raise ValueError(f"{symbol}: unexpected response {payload}")
        if not payload:
            print(f"{symbol}: no additional candles returned.")
            break

        b = parse_batch(payload, symbol)
        b = b.loc[b["open_time_ms"] <= effective_last_open_ms].copy()
        if b.empty:
            break

        last_open = int(b["open_time_ms"].iloc[-1])
        if last_open < cursor:
            raise RuntimeError(f"{symbol}: pagination did not advance.")

        batches.append(b)
        batch_no += 1
        new_rows += len(b)
        cursor = last_open + INTERVAL_MS

        if batch_no % CHECKPOINT_EVERY_BATCHES == 0:
            save_checkpoint(pd.concat([existing, *batches], ignore_index=True), symbol)
            print(
                f"{symbol}: checkpoint | batches={batch_no:,} | "
                f"new rows={new_rows:,} | "
                f"last={pd.to_datetime(last_open, unit='ms', utc=True)}"
            )

        if last_open >= effective_last_open_ms:
            break

        time.sleep(REQUEST_SLEEP_SECONDS)

    final = (
        pd.concat([existing, *batches], ignore_index=True)
        if batches else existing.copy()
    )
    final = (
        final.sort_values("open_time_ms")
             .drop_duplicates("open_time_ms", keep="last")
             .reset_index(drop=True)
    )
    save_checkpoint(final, symbol)
    print(f"{symbol}: final rows={len(final):,}")
    return final

## Pull all four assets

In [6]:
symbol_frames = {}
started = time.time()

for symbol in SYMBOLS:
    print("\n" + "="*80)
    print(f"{symbol} | {KLINE_INTERVAL}")
    print("="*80)
    symbol_frames[symbol] = pull_symbol(symbol)

print(f"\nElapsed minutes: {(time.time()-started)/60:.2f}")


BTCUSDT | 15m
BTCUSDT: existing checkpoint begins too late; restarting.


/tmp/ipykernel_5433/265029496.py:125: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  save_checkpoint(pd.concat([existing, *batches], ignore_index=True), symbol)


BTCUSDT: checkpoint | batches=50 | new rows=50,000 | last=2019-01-23 01:30:00+00:00


/tmp/ipykernel_5433/265029496.py:125: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  save_checkpoint(pd.concat([existing, *batches], ignore_index=True), symbol)


BTCUSDT: checkpoint | batches=100 | new rows=100,000 | last=2020-06-28 17:30:00+00:00


...

XRPUSDT: checkpoint | batches=200 | new rows=200,000 | last=2024-01-20 17:00:00+00:00


/tmp/ipykernel_5433/265029496.py:125: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  save_checkpoint(pd.concat([existing, *batches], ignore_index=True), symbol)


XRPUSDT: checkpoint | batches=250 | new rows=250,000 | last=2025-06-24 13:00:00+00:00


/tmp/ipykernel_5433/265029496.py:138: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([existing, *batches], ignore_index=True)


XRPUSDT: final rows=288,731

Elapsed minutes: 4.30


## Combine and audit the raw dataset

In [7]:
raw_df = (
    pd.concat(
        [symbol_frames[s] for s in SYMBOLS],
        ignore_index=True
    )
    .sort_values(
        ["symbol", "open_time_ms"]
    )
    .drop_duplicates(
        ["symbol", "open_time_ms"],
        keep="last"
    )
    .reset_index(drop=True)
)


# ============================================================
# 1. Schema / duplicates / basic structure
# ============================================================

assert list(raw_df.columns) == FINAL_COLUMNS

assert not raw_df.duplicated(
    ["symbol", "open_time_ms"]
).any()

assert (
    raw_df["symbol"].nunique()
    == len(SYMBOLS)
)

assert (
    raw_df["interval"]
    .eq(KLINE_INTERVAL)
    .all()
)


# ============================================================
# 2. Missing-value checks
# ============================================================

critical_columns = [
    "open_time",
    "close_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "quote_asset_volume",
    "number_of_trades",
]

critical_null_counts = (
    raw_df[
        critical_columns
    ]
    .isna()
    .sum()
)

display(
    critical_null_counts
    .rename("null_count")
    .to_frame()
)

assert (
    critical_null_counts
    == 0
).all()


# ============================================================
# 3. OPEN-TIME STRUCTURE
#
# This is the timestamp structure that matters for:
# - returns
# - continuity
# - rolling features
# - t -> t+1 target creation
# ============================================================

open_alignment_mask = (
    raw_df["open_time_ms"]
    % INTERVAL_MS
    != 0
)

open_alignment_mismatches = int(
    open_alignment_mask.sum()
)

print(
    "Open-time grid mismatches:",
    open_alignment_mismatches
)

# Fixed Binance intervals should open exactly on the grid.
assert (
    open_alignment_mismatches
    == 0
)


# ============================================================
# 4. CLOSE-TIME STRUCTURE AUDIT
#
# We DO NOT require close_time to be exactly:
#
# open_time + INTERVAL_MS - 1
#
# for every historical Binance row.
#
# close_time is preserved exactly as returned by Binance.
# Continuity is determined from open_time instead.
# ============================================================

expected_close_ms = (
    raw_df["open_time_ms"]
    + INTERVAL_MS
    - 1
)

close_mismatch_mask = (
    raw_df["close_time_ms"]
    != expected_close_ms
)


close_time_audit_df = (
    raw_df.loc[
        close_mismatch_mask,
        [
            "symbol",
            "interval",
            "open_time",
            "close_time",
            "open_time_ms",
            "close_time_ms",
        ],
    ]
    .copy()
)


close_time_audit_df[
    "expected_close_time_ms"
] = (
    close_time_audit_df[
        "open_time_ms"
    ]
    + INTERVAL_MS
    - 1
)


close_time_audit_df[
    "actual_candle_duration_ms"
] = (
    close_time_audit_df[
        "close_time_ms"
    ]
    - close_time_audit_df[
        "open_time_ms"
    ]
    + 1
)


close_time_audit_df[
    "close_time_delta_ms"
] = (
    close_time_audit_df[
        "close_time_ms"
    ]
    - close_time_audit_df[
        "expected_close_time_ms"
    ]
)


close_time_audit_df.to_csv(
    CANDLE_STRUCTURE_AUDIT_PATH,
    index=False
)


print(
    "Close-time metadata mismatches:",
    len(close_time_audit_df)
)


if not close_time_audit_df.empty:

    display(
        close_time_audit_df.head(20)
    )

    print(
        "\nDistribution of close-time "
        "difference from expected:"
    )

    display(
        close_time_audit_df[
            "close_time_delta_ms"
        ]
        .value_counts()
        .sort_index()
        .rename("rows")
        .to_frame()
    )

    warnings.warn(
        f"{len(close_time_audit_df)} candles have "
        "non-standard Binance close_time metadata. "
        "They are preserved and audited. "
        "Open timestamps will be used for "
        "continuity and modelling."
    )

else:

    print(
        "All close timestamps match "
        "the nominal interval structure."
    )


# The only genuinely invalid close-time situation
# would be a candle reported as closing before it opened.
invalid_close_order = (
    raw_df["close_time_ms"]
    < raw_df["open_time_ms"]
)

assert not (
    invalid_close_order.any()
)


# ============================================================
# 5. OHLC consistency
# ============================================================

assert (
    raw_df["high"]
    .ge(
        raw_df[
            [
                "open",
                "close",
                "low",
            ]
        ]
        .max(axis=1)
    )
    .all()
)

assert (
    raw_df["low"]
    .le(
        raw_df[
            [
                "open",
                "close",
                "high",
            ]
        ]
        .min(axis=1)
    )
    .all()
)

assert (
    raw_df["high"]
    .ge(
        raw_df["low"]
    )
    .all()
)


# ============================================================
# 6. Non-negative activity fields
# ============================================================

for column in [
    "volume",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_asset_volume",
    "taker_buy_quote_asset_volume",
]:

    invalid_count = int(
        (
            raw_df[column]
            < 0
        ).sum()
    )

    if invalid_count > 0:

        raise ValueError(
            f"{column} contains "
            f"{invalid_count} negative rows."
        )


# ============================================================
# 7. No unfinished candle
# ============================================================

assert (
    raw_df["open_time_ms"]
    <= effective_last_open_ms
).all()


display(
    raw_df.head()
)

print(
    "Combined shape:",
    raw_df.shape
)

print(
    "\nStructural QC passed."
)

print(
    "IMPORTANT: sequence continuity "
    "will be based on open_time_ms."
)

,null_count
open_time,0
close_time,0
open,0
high,0
low,0
close,0
volume,0
quote_asset_volume,0
number_of_trades,0


Open-time grid mismatches: 0
Close-time metadata mismatches: 46


,symbol,interval,open_time,close_time,open_time_ms,close_time_ms,expected_close_time_ms,actual_candle_duration_ms,close_time_delta_ms
1968,BTCUSDT,15m,2017-09-06 16:00:00+00:00,2017-09-06 16:00:00+00:00,1504713600000,1504713600000,1504714499999,1,-899999
10445,BTCUSDT,15m,2017-12-04 06:00:00+00:00,2017-12-04 06:00:20.798000+00:00,1512367200000,1512367220798,1512368099999,20799,-879201
11812,BTCUSDT,15m,2017-12-18 12:15:00+00:00,2017-12-18 12:29:13.419000+00:00,1513599300000,1513600153419,1513600199999,853420,-46580
13403,BTCUSDT,15m,2018-01-04 03:00:00+00:00,2018-01-04 03:00:14.838000+00:00,1515034800000,1515034814838,1515035699999,14839,-885161
16745,BTCUSDT,15m,2018-02-08 00:15:00+00:00,2018-02-08 00:28:14.788000+00:00,1518048900000,1518049694788,1518049799999,794789,-105211
16913,BTCUSDT,15m,2018-02-11 04:00:00+00:00,2018-02-11 04:00:00.999000+00:00,1518321600000,1518321600999,1518322499999,1000,-899000
30578,BTCUSDT,15m,2018-07-04 00:15:00+00:00,2018-07-04 00:22:25.551000+00:00,1530663300000,1530663745551,1530664199999,445552,-454448
62973,BTCUSDT,15m,2019-06-07 21:00:00+00:00,2019-06-07 21:13:13.524000+00:00,1559941200000,1559941993524,1559942099999,793525,-106475
87550,BTCUSDT,15m,2020-02-19 11:30:00+00:00,2020-02-19 11:35:32.286000+00:00,1582111800000,1582112132286,1582112699999,332287,-567713
88862,BTCUSDT,15m,2020-03-04 09:15:00+00:00,2020-03-04 09:21:46.694000+00:00,1583313300000,1583313706694,1583314199999,406695,-493305



Distribution of close-time difference from expected:


,rows
close_time_delta_ms,
-899999,2
-899000,2
-885161,1
-885150,1
-879201,1
-879190,1
-841853,1
-840728,1
-838460,1


/tmp/ipykernel_5433/2801778542.py:217: UserWarning: 46 candles have non-standard Binance close_time metadata. They are preserved and audited. Open timestamps will be used for continuity and modelling.
  warnings.warn(


,symbol,interval,open_time,close_time,open_time_ms,close_time_ms,open,high,low,close,volume,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume
0,BTCUSDT,15m,2017-08-17 04:00:00+00:00,2017-08-17 04:14:59.999000+00:00,1502942400000,1502943299999,4261.48,4280.56,4261.48,4261.48,2.189061,9333.620962,9,0.489061,2089.104962
1,BTCUSDT,15m,2017-08-17 04:15:00+00:00,2017-08-17 04:29:59.999000+00:00,1502943300000,1502944199999,4261.48,4270.41,4261.32,4261.45,9.119865,38891.133046,40,3.447113,14703.934995
2,BTCUSDT,15m,2017-08-17 04:30:00+00:00,2017-08-17 04:44:59.999000+00:00,1502944200000,1502945099999,4280.00,4310.07,4267.99,4310.07,21.923552,94080.917568,58,20.421317,87620.977876
3,BTCUSDT,15m,2017-08-17 04:45:00+00:00,2017-08-17 04:59:59.999000+00:00,1502945100000,1502945999999,4310.07,4313.62,4291.37,4308.83,13.948531,60060.466816,64,10.803012,46538.460109
4,BTCUSDT,15m,2017-08-17 05:00:00+00:00,2017-08-17 05:14:59.999000+00:00,1502946000000,1502946899999,4308.83,4328.69,4304.31,4304.31,5.101153,22006.533111,44,3.496635,15093.783057


Combined shape: (1125147, 15)

Structural QC passed.
IMPORTANT: sequence continuity will be based on open_time_ms.


In [8]:
raw_df[~(raw_df["close_time_ms"].eq(expected_close_ms))].count()

,0
symbol,46
interval,46
open_time,46
close_time,46
open_time_ms,46
close_time_ms,46
open,46
high,46
low,46
close,46


## Continuity audit

Missing bars are **reported, not filled**. Notebook 02 must only compute returns/features across consecutive candles.

In [9]:
gap_rows = []

for symbol in SYMBOLS:
    f = (
        raw_df.loc[raw_df["symbol"].eq(symbol)]
              .sort_values("open_time_ms")
              .reset_index(drop=True)
    )

    diff = f["open_time_ms"].diff()
    gap_mask = diff.notna() & diff.ne(INTERVAL_MS)
    positions = np.where(gap_mask.to_numpy())[0]

    total_missing = 0
    max_gap_intervals = 1
    first_prev = pd.NaT
    first_next = pd.NaT

    for pos in positions:
        interval_count = int(diff.iloc[pos] // INTERVAL_MS)
        total_missing += max(interval_count - 1, 0)
        max_gap_intervals = max(max_gap_intervals, interval_count)
        if pd.isna(first_prev):
            first_prev = f["open_time"].iloc[pos-1]
            first_next = f["open_time"].iloc[pos]

    first_ms = int(f["open_time_ms"].min())
    last_ms = int(f["open_time_ms"].max())
    expected = (last_ms - first_ms) // INTERVAL_MS + 1

    gap_rows.append({
        "symbol":symbol,
        "interval":KLINE_INTERVAL,
        "rows":len(f),
        "first_open_time":f["open_time"].min(),
        "last_open_time":f["open_time"].max(),
        "expected_rows_between_first_last":int(expected),
        "gap_events":int(len(positions)),
        "implied_missing_candles":int(total_missing),
        "max_gap_intervals":int(max_gap_intervals),
        "first_gap_previous_candle":first_prev,
        "first_gap_next_candle":first_next,
    })

gap_audit_df = pd.DataFrame(gap_rows)
display(gap_audit_df)
gap_audit_df.to_csv(GAP_AUDIT_PATH, index=False)

if gap_audit_df["gap_events"].sum() > 0:
    warnings.warn(
        "Historical gaps exist and were NOT filled. "
        "Notebook 02 must enforce consecutive-candle logic."
    )
else:
    print("No internal candle gaps found.")

,symbol,interval,rows,first_open_time,last_open_time,expected_rows_between_first_last,gap_events,implied_missing_candles,max_gap_intervals,first_gap_previous_candle,first_gap_next_candle
0,BTCUSDT,15m,313531,2017-08-17 04:00:00+00:00,2026-08-01 23:45:00+00:00,314096,33,565,135,2017-09-06 16:00:00+00:00,2017-09-06 23:00:00+00:00
1,ETHUSDT,15m,313531,2017-08-17 04:00:00+00:00,2026-08-01 23:45:00+00:00,314096,33,565,135,2017-09-06 16:00:00+00:00,2017-09-06 23:00:00+00:00
2,SOLUSDT,15m,209354,2020-08-11 06:00:00+00:00,2026-08-01 23:45:00+00:00,209448,10,94,19,2020-11-30 05:45:00+00:00,2020-11-30 07:00:00+00:00
3,XRPUSDT,15m,288731,2018-05-04 08:00:00+00:00,2026-08-01 23:45:00+00:00,289120,26,389,41,2018-06-26 01:45:00+00:00,2018-06-26 12:00:00+00:00


/tmp/ipykernel_5433/314545201.py:50: UserWarning: Historical gaps exist and were NOT filled. Notebook 02 must enforce consecutive-candle logic.
  warnings.warn(


## Save final parquet, reload it, and create manifest

In [10]:
asset_summary_df = (
    raw_df.groupby("symbol")
          .agg(
              rows=("open_time","size"),
              first_open_time=("open_time","min"),
              last_open_time=("open_time","max"),
              total_trades=("number_of_trades","sum"),
          )
          .reset_index()
)

display(asset_summary_df)

raw_df.to_parquet(COMBINED_RAW_PATH, index=False)

# Persistence QC
reloaded = pd.read_parquet(COMBINED_RAW_PATH)
assert len(reloaded) == len(raw_df)
assert list(reloaded.columns) == FINAL_COLUMNS
assert not reloaded.duplicated(["symbol","open_time_ms"]).any()
assert reloaded["symbol"].nunique() == len(SYMBOLS)
assert reloaded["interval"].eq(KLINE_INTERVAL).all()

def sha256_file(path, chunk_size=1024*1024):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            chunk = fh.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

per_asset = {}

for _, row in asset_summary_df.iterrows():
    symbol = row["symbol"]
    gap = gap_audit_df.loc[gap_audit_df["symbol"].eq(symbol)].iloc[0]
    per_asset[symbol] = {
        "rows":int(row["rows"]),
        "first_open_time_utc":row["first_open_time"].isoformat(),
        "last_open_time_utc":row["last_open_time"].isoformat(),
        "gap_events":int(gap["gap_events"]),
        "implied_missing_candles":int(gap["implied_missing_candles"]),
    }

manifest = {
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "project_stage":"01_intraday_data_creation",
    "source":"Binance Spot public market data",
    "base_url":BASE_URL,
    "endpoint":"/api/v3/klines",
    "symbols":SYMBOLS,
    "kline_interval":KLINE_INTERVAL,
    "interval_minutes":INTERVAL_MINUTES,
    "interval_ms":INTERVAL_MS,
    "request_limit":REQUEST_LIMIT,
    "requested_start_date_utc":START_DATE,
    "requested_end_date_utc_inclusive":END_DATE,
    "effective_first_open_time_utc":pd.to_datetime(
        first_requested_open_ms, unit="ms", utc=True
    ).isoformat(),
    "effective_last_completed_open_time_utc":pd.to_datetime(
        effective_last_open_ms, unit="ms", utc=True
    ).isoformat(),
    "combined_rows":int(len(raw_df)),
    "combined_output_path":str(COMBINED_RAW_PATH),
    "combined_file_size_bytes":int(COMBINED_RAW_PATH.stat().st_size),
    "combined_sha256":sha256_file(COMBINED_RAW_PATH),
    "gap_audit_path":str(GAP_AUDIT_PATH),
    "per_asset":per_asset,
    "raw_data_policy": {

    "fill_missing_candles":
        False,

    "drop_incomplete_current_candle":
        True,

    "deduplicate_by": [
        "symbol",
        "open_time_ms",
    ],

    "timezone":
        "UTC",

    "continuity_timestamp":
        "open_time_ms",

    "require_open_time_grid_alignment":
        True,

    "close_time_policy":
        "preserve_binance_value_and_audit",

    "require_exact_close_time":
        False,
},
    "candle_structure_audit_path":
    str(
        CANDLE_STRUCTURE_AUDIT_PATH
    ),

"close_time_structure_mismatches":
    int(
        len(
            close_time_audit_df
        )
    ),
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as fh:
    json.dump(manifest, fh, indent=2, ensure_ascii=False)

assert COMBINED_RAW_PATH.exists()
assert GAP_AUDIT_PATH.exists()
assert CANDLE_STRUCTURE_AUDIT_PATH.exists()
assert MANIFEST_PATH.exists()

print("Saved:", COMBINED_RAW_PATH)
print("Gap audit:", GAP_AUDIT_PATH)
print("Manifest:", MANIFEST_PATH)
print("SHA256:", manifest["combined_sha256"])
print("\nALL RAW-DATA CREATION QC CHECKS PASSED.")

,symbol,rows,first_open_time,last_open_time,total_trades
0,BTCUSDT,313531,2017-08-17 04:00:00+00:00,2026-08-01 23:45:00+00:00,6550994183
1,ETHUSDT,313531,2017-08-17 04:00:00+00:00,2026-08-01 23:45:00+00:00,4249920343
2,SOLUSDT,209354,2020-08-11 06:00:00+00:00,2026-08-01 23:45:00+00:00,2016771708
3,XRPUSDT,288731,2018-05-04 08:00:00+00:00,2026-08-01 23:45:00+00:00,1669002446


Saved: /content/drive/MyDrive/Quant Research/15min/data/raw/15m/crypto_binance_15m_btc_eth_sol_xrp_raw.parquet
Gap audit: /content/drive/MyDrive/Quant Research/15min/manifests/15m/gap_audit_15m.csv
Manifest: /content/drive/MyDrive/Quant Research/15min/manifests/15m/crypto_binance_15m_btc_eth_sol_xrp_raw_manifest.json
SHA256: 03e2255ead08f8f9be92958ad073024edfe2bfb155be553da3eb06c3ba61026b

ALL RAW-DATA CREATION QC CHECKS PASSED.


## Next step

Notebook 02 will start from this saved raw parquet and:

1. calculate log returns only across consecutive candles;
2. define next-candle realized variance \(RV_{t+1}=r_{t+1}^2\);
3. create leakage-safe rolling features;
4. add intraday periodicity/time-of-day features;
5. examine intraday volatility seasonality;
6. create Train / Validation / Test splits by **target timestamp**;
7. prepare the model-ready arrays used by ARCH and DL.

Keeping all transformation logic out of Notebook 01 means this raw-data layer remains reusable if `KLINE_INTERVAL` later changes from `15m` to `1h`.